# Recommendation system

Suggest relevant products, content or people to each user.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/recommendation/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

Popularity + content-based similarity (embeddings of item descriptions): Works with little interaction data and handles new items.

In [ ]:
%pip install -q sentence-transformers pandas

In [ ]:
# Content-based "similar items"
# pip install sentence-transformers pandas
import pandas as pd
from sentence_transformers import SentenceTransformer, util

items = pd.read_csv("items.csv")            # id, title, description
model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode((items.title + ". " + items.description).tolist(), convert_to_tensor=True)
scores = util.cos_sim(emb[0], emb)[0]
print(items.iloc[scores.argsort(descending=True)[1:6].cpu()].title)

## Train your own model

Collaborative filtering (implicit ALS / LightFM) + content features: Learns taste from behavior; proven and cheap to run.

In [ ]:
%pip install -q implicit pandas scipy

In [ ]:
# pip install implicit pandas scipy
import pandas as pd, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

ev = pd.read_csv("events.csv")              # user_id, item_id, weight
users = ev.user_id.astype("category"); items = ev.item_id.astype("category")
mat = sp.csr_matrix((ev.weight, (users.cat.codes, items.cat.codes)))

model = AlternatingLeastSquares(factors=64, iterations=20)
model.fit(mat)
uid = 0
ids, scores = model.recommend(uid, mat[uid], N=10)
print([items.cat.categories[i] for i in ids])